# 03 — Model Training

Interactive walkthrough of dataset construction, model building, and two-stage training. For full runs prefer the CLI: `python train.py`.

In [ ]:
import sys
sys.path.append('..')
from config import CONFIG, ensure_directories
from src.dataset import PneumoniaDataset
from src.model import build_model
from src.trainer import Trainer
from src.utils import set_seed, get_logger

ensure_directories()
set_seed(CONFIG.train.seed)
logger = get_logger('notebook_train', CONFIG.paths.logs_dir)

## Build datasets

In [ ]:
train_data = PneumoniaDataset(CONFIG.paths.train_dir, image_size=CONFIG.data.image_size, batch_size=CONFIG.data.batch_size)
val_data = PneumoniaDataset(CONFIG.paths.val_dir, image_size=CONFIG.data.image_size, batch_size=CONFIG.data.batch_size)
print('Train images:', len(train_data), '| Val images:', len(val_data))
print('Class weights:', train_data.class_weights)

## Build the model

In [ ]:
model = build_model(
    input_shape=CONFIG.model.input_shape,
    dropout_1=CONFIG.model.dropout_1,
    dropout_2=CONFIG.model.dropout_2,
    dense_1=CONFIG.model.dense_1,
    dense_2=CONFIG.model.dense_2,
    freeze_backbone=CONFIG.model.freeze_backbone,
)
model.summary()

## Train (two stages: frozen backbone, then fine-tune)

This cell requires `dataset/train` and `dataset/val` to be populated. Skip in an environment without the data / a GPU.

In [ ]:
# train_ds = train_data.build(training=True)
# val_ds = val_data.build(training=False)
# trainer = Trainer(model, CONFIG, logger=logger)
# trainer.fit(train_ds, val_ds, class_weight=train_data.class_weights, fine_tune=True)

## Plot training curves

Run after `trainer.fit(...)` above.

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['loss'], label='train')
    axes[0].plot(history.history['val_loss'], label='val')
    axes[0].set_title(f'{title} — Loss'); axes[0].legend()
    axes[1].plot(history.history['auc'], label='train')
    axes[1].plot(history.history['val_auc'], label='val')
    axes[1].set_title(f'{title} — AUC'); axes[1].legend()
    plt.show()

# plot_history(trainer.history_stage1, 'Stage 1 (frozen backbone)')
# plot_history(trainer.history_stage2, 'Stage 2 (fine-tuning)')